In [3]:
import pandas as pd
import numpy as np
import joblib
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import accuracy_score, classification_report, mean_absolute_error, r2_score

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent

df = pd.read_csv(root / "data" / "cleaned_posts.csv")

feature_cols = ["post_type", "topic", "video_duration",
                "posting_hour", "hashtags_count", "caption_length"]

# Turn text columns (post_type, topic) into 0/1 columns the model can use
X = pd.get_dummies(df[feature_cols], columns=["post_type", "topic"])
y_class = df["performance_level"]
y_reg = df["reach_ratio"]

print(X.shape)
X.head()

(300, 14)


,video_duration,posting_hour,hashtags_count,caption_length,post_type_Carousel,post_type_Image,post_type_Reel,topic_Career Advice,topic_Data Analysis,topic_Excel Tricks,topic_Machine Learning Basics,topic_Python Interview Tips,topic_Python Tips,topic_SQL Tutorial
0,21,18,6,85,False,False,True,True,False,False,False,False,False,False
1,56,9,6,233,False,False,True,False,False,False,True,False,False,False
2,0,18,7,89,False,True,False,True,False,False,False,False,False,False
3,0,9,10,187,True,False,False,False,False,True,False,False,False,False
4,0,12,13,108,True,False,False,True,False,False,False,False,False,False


In [4]:
X_train, X_test, yc_train, yc_test, yr_train, yr_test = train_test_split(
    X, y_class, y_reg, test_size=0.2, random_state=42, stratify=y_class
)

clf = RandomForestClassifier(n_estimators=200, random_state=42)
clf.fit(X_train, yc_train)

pred = clf.predict(X_test)
print("Accuracy:", round(accuracy_score(yc_test, pred), 3))
print(classification_report(yc_test, pred))

Accuracy: 0.65
              precision    recall  f1-score   support

        High       0.73      0.55      0.63        20
         Low       0.73      0.80      0.76        20
      Medium       0.52      0.60      0.56        20

    accuracy                           0.65        60
   macro avg       0.66      0.65      0.65        60
weighted avg       0.66      0.65      0.65        60



In [5]:
reg = RandomForestRegressor(n_estimators=200, random_state=42)
reg.fit(X_train, yr_train)

rp = reg.predict(X_test)
print("R2 score:", round(r2_score(yr_test, rp), 3))
print("Average error (reach ratio):", round(mean_absolute_error(yr_test, rp), 3))

R2 score: 0.569
Average error (reach ratio): 0.423


In [6]:
importance = pd.Series(clf.feature_importances_, index=X.columns).sort_values(ascending=False)
print(importance.head(8).round(3))

video_duration                   0.183
caption_length                   0.178
posting_hour                     0.149
hashtags_count                   0.134
post_type_Reel                   0.118
post_type_Image                  0.073
post_type_Carousel               0.037
topic_Machine Learning Basics    0.027
dtype: float64


In [7]:
joblib.dump(clf, root / "models" / "performance_classifier.pkl")
joblib.dump(reg, root / "models" / "reach_regressor.pkl")
joblib.dump(list(X.columns), root / "models" / "feature_columns.pkl")
print("Models saved")

Models saved
